# Gemini API
**Materi ini dibuat oleh:** [Sardi Irfansyah](https://www.linkedin.com/in/sirfansyah/)

> Converted for local VS Code usage. Requires Python 3.9+ and a Gemini API Key.

## Setup
1. Install dependencies: `pip install -r requirements.txt`
2. Create a `.env` file in this folder:
   ```
   GEMINI_API_KEY=your_api_key_here
   ```
3. Get your API key from [Google AI Studio](https://aistudio.google.com/)

## Instalasi

Sebelum memulai, install package yang diperlukan:

In [ ]:
%pip install -q -U google-generativeai python-dotenv Pillow httpx

## Load API Key

In [ ]:
import os
import base64
import urllib.request
import httpx
import PIL.Image
import google.generativeai as genai
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found. Please create a .env file with your API key.")

genai.configure(api_key=api_key)
print("API Key loaded successfully!")

---
## 1. Text Generation

Gunakan metode `generate_content` untuk mengirim permintaan ke API Gemini.

In [ ]:
model = genai.GenerativeModel("gemini-2.5-flash")
response = model.generate_content("Explain how AI works")
display(Markdown(response.text))

Untuk mengecek list model yang tersedia pada Gemini:

In [ ]:
for m in genai.list_models():
    print(m.name)

---
## 2. Streaming Response

Gunakan `stream=True` untuk mendapatkan respons secara bertahap (lebih cepat terasa).

In [ ]:
response = model.generate_content("Explain how AI works", stream=True)
for chunk in response:
    print(chunk.text, end="")

---
## 3. Image Input

Kita akan menggunakan PIL untuk membaca gambar kemudian menambahkan prompt untuk menghasilkan teks.

In [ ]:
image_url = "https://marketplace.canva.com/EAFhLHB9S8g/1/0/900w/canva-pink-putih-minimalist-salon-price-list-28N6AZMRZKI.jpg"
urllib.request.urlretrieve(image_url, "salon_price_list.jpg")
print("Image downloaded: salon_price_list.jpg")

In [ ]:
img = PIL.Image.open("salon_price_list.jpg")

# Tanpa prompt
response = model.generate_content(["", img])
display(Markdown(response.text))

In [ ]:
# Dengan prompt: buat tabel
response = model.generate_content(["buatlah table dari response tersebut", img])
display(Markdown(response.text))

In [ ]:
# Dengan prompt: buat tabel + kode Python DataFrame
prompt = """buatlah tabel dari response tersebut,
buatlah code python untuk membuat dataframe dari tabel tersebut"""
response = model.generate_content([prompt, img])
display(Markdown(response.text))

---
## 4. Document Understanding (PDF)

In [ ]:
doc_url = "https://discovery.ucl.ac.uk/id/eprint/10089234/1/343019_3_art_0_py4t4l_convrt.pdf"
doc_data = base64.standard_b64encode(httpx.get(doc_url).content).decode("utf-8")

response = model.generate_content([{"mime_type": "application/pdf", "data": doc_data}, "Summarize this document"])
display(Markdown(response.text))

---
## 5. Audio Understanding

In [ ]:
audio_url = "https://storage.googleapis.com/generativeai-downloads/data/State_of_the_Union_Address_30_January_1961.mp3"
urllib.request.urlretrieve(audio_url, "sample.mp3")
print("Audio downloaded: sample.mp3")

In [ ]:
myfile = genai.upload_file("sample.mp3")
print(f"{myfile=}")

result = model.generate_content([myfile, "Describe this audio clip"])
display(Markdown(result.text))

---
## 6. Chat Conversation

Gemini SDK mendukung multi-turn conversation dengan melacak **riwayat percakapan**.

In [ ]:
chat = model.start_chat(history=[])

response = chat.send_message("Dalam satu kalimat, jelaskan cara kerja komputer kepada anak kecil.")
display(Markdown(response.text))

In [ ]:
response = chat.send_message("Jelaskan cara membuat otomatisasi email")
display(Markdown(response.text))

In [ ]:
# Cek history percakapan
for message in chat.history:
    print(f"[{message.role.upper()}]")
    print(message.parts[0].text[:200], "...\n")

In [ ]:
# Chat dengan pre-defined history
chat = model.start_chat(
    history=[
        {"role": "user", "parts": "Hello"},
        {"role": "model", "parts": "Great to meet you. What would you like to know?"},
    ]
)

response = chat.send_message("I have 2 dogs in my house.")
print(response.text)

response2 = chat.send_message("How many paws are in my house?")
print(response2.text)

---
## 7. Latihan

Cobalah beberapa teknik prompting dasar yang telah dipelajari:
- Zero-Shot Prompting
- One-Shot Prompting
- Few-Shot Prompting
- Role-Based Prompting

In [ ]:
# Tulis kode latihan Anda di sini
